Step 1: Check MLflow and import ResponsesAgent

In [0]:
import mlflow

print("MLflow version:", mlflow.__version__)

from mlflow.pyfunc import ResponsesAgent

print("ResponsesAgent imported successfully!")

In [0]:
from mlflow.pyfunc import ResponsesAgent

class MyHRAgent(ResponsesAgent):
    pass

print("MyHRAgent class created successfully!")

What are we doing here?

class MyHRAgent → Creating our own agent.

(ResponsesAgent) → Inheriting from the MLflow base class.

pass → Keeping the class empty for now. We haven't implemented its behaviour yet.

predict() receives the user's request and returns the agent's response.


For now, we'll keep it simple. No LLM or tools yet. We just want to understand how the request enters and the response comes out.

In [0]:
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse
)

class MyHRAgent(ResponsesAgent):

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:

        # Our simple agent response
        answer = "Hello! I am your Company HR Assistant."

        return ResponsesAgentResponse(
            output=[
                self.create_text_output_item(
                    text=answer,
                    id="msg_1"
                )
            ]
        )

print("Custom HR Agent created successfully!")

Understand what we wrote

Code

	

Purpose




predict()

	

Receives the request.




request

	

Contains the user's input.




answer

	

Our agent's response.




create_text_output_item()

	

Creates a standard text output item.




ResponsesAgentResponse()

	

Packages the output in the expected response format.

Important: Right now, our agent returns a fixed answer. It doesn't understand the user's question yet. We'll add that intelligence later.

In [0]:
agent = MyHRAgent()

request = ResponsesAgentRequest(
    input=[
        {
            "role": "user",
            "content": "Hi, who are you?"
        }
    ]
)

response = agent.predict(request)

print(response)

PREVIOUS ONE WAS PRACTICE WITHOUT LLM 

Connect the LLM to MyHRAgent

reviously, our agent returned a fixed answer:

answer = "Hello! I am your Company HR Assistant."

Now, instead of a fixed answer, we want the LLM to generate an answer based on the user's question.


We'll use the Databricks SDK to connect our existing MyHRAgent to the Llama 4 Maverick endpoint.

In [0]:
from databricks.sdk import WorkspaceClient

client = WorkspaceClient()

print("Databricks SDK client created successfully!")    

"Databricks SDK is a Python software development kit that allows developers to programmatically interact with Databricks workspace resources and services through APIs."

In [0]:
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

response = client.serving_endpoints.query(
    name="databricks-llama-4-maverick",
    messages=[
        ChatMessage(
            role=ChatMessageRole.USER,
            content="Explain Databricks in two simple sentences."
        )
    ],
    max_tokens=150  
)

print(response)

Both ChatMessage and ChatMessageRole are provided by the Databricks SDK for Python.


ChatMessage

	

Represents a message sent to or received from the model.




ChatMessageRole

	

Identifies who sent the message: USER, ASSISTANT, etc.

### Implement the LLM inside `ResponsesAgent`

In [0]:
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse
)
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole


class MyHRAgent(ResponsesAgent):

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:

        # Step 1: Extract the user's question
        user_message = request.input[-1].content

        # Step 2: Send the question to Llama
        response = client.serving_endpoints.query(
            name="databricks-llama-4-maverick",
            messages=[
                ChatMessage(
                    role=ChatMessageRole.USER,
                    content=str(user_message)
                )
            ],
            max_tokens=200
        )

        # Step 3: Extract Llama's answer
        answer = response.choices[0].message.content

        # Step 4: Return the answer in ResponsesAgent format
        return ResponsesAgentResponse(
            output=[
                self.create_text_output_item(
                    text=answer,
                    id="msg_1"
                )
            ]
        )


print("LLM-powered HR Agent created successfully!")

Important: We haven't added tool calling yet. At this stage, our agent can answer questions using Llama, but it cannot retrieve leave balances from our Unity Catalog function.

Create an agent instance and send a question
What is an instance?

We have created the agent's class, which is like a blueprint. Now we need to create an actual agent object from that blueprint.

In [0]:
agent = MyHRAgent()

request = ResponsesAgentRequest(
    input=[
        {
            "role": "user",
            "content": "What is Unity Catalog? Explain in three simple points."
        }
    ]
)

response = agent.predict(request)

print(response.output[0].content)

### How will our HR Agent use a Unity Catalog function?![](path)

In [0]:
%pip install "databricks-langchain==0.5.0" "unitycatalog-langchain[databricks]==0.4.0"

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks_langchain import UCFunctionToolkit

print("UCFunctionToolkit imported successfully!")

create the Unity Catalog tool

In [0]:
from databricks_langchain import UCFunctionToolkit

function_name = "sona_agent_lab.hr_tools.get_leave_balance"

toolkit = UCFunctionToolkit(
    function_names=[function_name]
)

tools = toolkit.tools

print("Unity Catalog tool loaded successfully!")

Initialize the LLM

In [0]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(
    endpoint="databricks-llama-4-maverick",
    temperature=0.1,
    max_tokens=300
)

print("LLM initialized successfully!")

In [0]:
llm_with_tools = llm.bind_tools(tools)

print("LLM connected with UC tools!")

We're telling the LLM:

"You have access to get_leave_balance. If a user's question requires an employee's leave balance, you can request this tool and provide the employee ID."

In [0]:
tool_response = llm_with_tools.invoke(
    "How many annual leave days does employee EMP101 have remaining?"
)

print(tool_response.tool_calls)

In [0]:
tool_call = tool_response.tool_calls[0]

tool_result = tools[0].invoke(
    tool_call["args"]
)

print("Tool result:", tool_result)

Our Unity Catalog function executed successfully and returned the correct leave balance: 12 days.

We'll use ToolMessage to send the result back to the LLM. This is the missing part of the tool-calling cycle.

In [0]:
from langchain_core.messages import ToolMessage

tool_message = ToolMessage(
    content=str(tool_result),
    tool_call_id=tool_call["id"]
)

print(tool_message)

In [0]:
final_response = llm_with_tools.invoke([
    {
        "role": "user",
        "content": "How many annual leave days does EMP101 have remaining?"
    },
    tool_response,
    tool_message
])

print(final_response.content)

## Build the complete HR ResponsesAgent

In [0]:
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse
)
from langchain_core.messages import ToolMessage


class MyHRAgent(ResponsesAgent):

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:

        # Step 1: Get the user's question
        user_message = request.input[-1].content

        # Step 2: Ask the LLM whether a tool is needed
        llm_response = llm_with_tools.invoke(
            str(user_message)
        )

        # Step 3: Check whether the LLM requested a tool
        if not llm_response.tool_calls:

            answer = llm_response.content

        else:

            # Step 4: Execute the requested tool
            tool_messages = []

            for tool_call in llm_response.tool_calls:

                selected_tool = next(
                    tool for tool in tools
                    if tool.name == tool_call["name"]
                )

                tool_result = selected_tool.invoke(
                    tool_call["args"]
                )

                # Step 5: Send the tool result back to the LLM
                tool_messages.append(
                    ToolMessage(
                        content=str(tool_result),
                        tool_call_id=tool_call["id"]
                    )
                )

            # Step 6: Ask the LLM to generate the final answer
            final_response = llm_with_tools.invoke([
                {
                    "role": "user",
                    "content": str(user_message)
                },
                llm_response,
                *tool_messages
            ])

            answer = final_response.content

        # Step 7: Return the answer in ResponsesAgent format
        return ResponsesAgentResponse(
            output=[
                self.create_text_output_item(
                    text=str(answer),
                    id="msg_1"
                )
            ]
        )


print("Complete HR ResponsesAgent created successfully!")

In [0]:
agent = MyHRAgent()
print("Agent instance created!")

WITHOUT TOOL

In [0]:
request = ResponsesAgentRequest(
    input=[
        {
            "role": "user",
            "content": "What is Unity Catalog in Databricks? Explain simply."
        }
    ]
)

response = agent.predict(request)

print(response.output[0].content)

WITH TOOL

In [0]:
request = ResponsesAgentRequest(
    input=[
        {
            "role": "user",
            "content": "How many annual leave days does EMP101 have remaining?"
        }
    ]
)

response = agent.predict(request)

print(response.output[0].content)